# 27. Automatic narrow-resonance grid refinement

**Objectives:**
- Configure `DecayModel(normalization_narrow_width=..., normalization_narrow_window=...,
  normalization_binning_factor=...)`, the knobs controlling automatic local refinement
  of the deterministic normalization quadrature.
- Build one model with a narrow resonance (width at or below the 20 MeV default
  threshold) and one with a broad resonance, and inspect the public
  `model.normalization_scheme` property on each.
- See in markdown why a narrow peak needs finer local sampling than a broad one under
  deterministic (non-adaptive-Monte-Carlo) quadrature.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

from dalitzplotfitter import DecayChannel, DecayModel, RealImag, Resonance

## 1. Why narrow resonances need special treatment

`DecayModel`'s default `normalization_method="gauss-legendre"` integrates the coherent
intensity with a fixed quadrature grid over the mass plane. A resonance whose width is
a small fraction of the grid spacing can fall almost entirely between grid nodes, so
the deterministic sum silently under- or over-estimates its contribution to the
normalization integral. `DecayModel` classifies a component "narrow" from its
*declared/initial* mass and width (not floated values) whenever
`0 < width <= normalization_narrow_width` (default 0.020 GeV), and then locally
refines the grid to a target spacing of `width / normalization_binning_factor` inside
a `+/- normalization_narrow_window * width` band around the pole.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

def make_model(width):
    return DecayModel(
        channel,
        [Resonance("R", pair=(1, 2), coefficient=RealImag(1.0, 0.0),
                   mass=1.0, width=width, spin=0),
         Resonance("NR_like", pair=(1, 2), coefficient=RealImag(0.3, 0.0),
                    mass=1.4, width=0.30, spin=0)],
        normalization_narrow_width=0.020,
        normalization_narrow_window=5.0,
        normalization_binning_factor=30.0,
    )

narrow_model = make_model(width=0.005)   # 5 MeV: below the 20 MeV threshold
broad_model = make_model(width=0.150)    # 150 MeV: above the threshold

## 2. Compare `normalization_scheme`

`normalization_scheme` is a public, read-only property describing the *active*
strategy without re-running the fit -- useful for confirming what a given
configuration will actually do before spending time on a full normalization.

In [3]:
narrow_scheme = narrow_model.normalization_scheme
broad_scheme = broad_model.normalization_scheme

print("Narrow-resonance model:")
for key, value in narrow_scheme.items():
    print(f"  {key}: {value}")

print("\nBroad-resonance model:")
for key, value in broad_scheme.items():
    print(f"  {key}: {value}")

assert narrow_scheme["adaptive"] is True
assert broad_scheme["adaptive"] is False
assert narrow_scheme["narrow_resonances"]["m23"], "expected the 5 MeV R to be classified narrow"
assert not broad_scheme.get("narrow_resonances", {}).get("m23"), "150 MeV should not trigger refinement"

Narrow-resonance model:
  method: gauss-legendre
  adaptive: True
  internal_coordinates: m13-m23
  narrow_resonances: {'m12': (), 'm13': (), 'm23': ((1.0, 0.005),)}
  m13_segments: (AdaptiveAxisSegment(low=0.27914078000000003, high=1.73008961, order=291, target_width=0.005, narrow=False),)
  m23_segments: (AdaptiveAxisSegment(low=0.27914078000000003, high=0.975, order=140, target_width=0.005, narrow=False), AdaptiveAxisSegment(low=0.975, high=1.025, order=300, target_width=0.00016666666666666666, narrow=True), AdaptiveAxisSegment(low=1.025, high=1.73008961, order=142, target_width=0.005, narrow=False))
  estimated_tensor_points: 169362

Broad-resonance model:
  method: gauss-legendre
  adaptive: False
  bin_width: 0.005
  order_m13: None
  order_m23: None


The narrow model's scheme reports `adaptive=True` with `internal_coordinates="m13-m23"`
and per-axis segment counts (`m13_segments`/`m23_segments`) built by
`AdaptiveDalitzGaussLegendreGrid` -- extra Gauss-Legendre panels concentrated in the
`+/- 5*Gamma` window around the 1.0 GeV, 5 MeV pole. The broad model's scheme is the
plain, non-adaptive `gauss-legendre` description: its 150 MeV width is wide enough
relative to the default fixed grid that no local refinement is triggered. Raising
`normalization_narrow_width` would let a wider resonance qualify as narrow;
raising `normalization_binning_factor` refines a narrow band's spacing further, at
the cost of more quadrature points inside that band.

## Continue learning

See the "Normalization" section of `README.md` and
[mc_integration.md](../../docs/mc_integration.md) for the full set of normalization
methods. Return to [the course guide](TUTORIALS.md).